# Visualization Quick Tour

**Part I · Visualization** — Tutorial 01

A rapid, example-driven tour through the `pytanga.viz` viewer — no
geometric-algebra background required. Each section glances at one major
capability and points to the chapter that covers it in full.

> **Note:** everything in this part uses geometry dataclasses from
> `pytanga.geometry` as plain 3D data. The viewer itself is introduced in
> [Tutorial 02](../02_getting_started/).


## Setup

The viewer lives in `pytanga.viz`; the geometry dataclasses live in
`pytanga.geometry`. In a notebook, `show()` renders inline; in a script it opens
a browser tab.


In [ ]:
from pytanga.geometry import Direction, Line, Plane, Point, Sphere
from pytanga.viz import (
    CoordinateSystem,
    LabelStyle,
    PointPathStyle,
    PointStyle,
    SdfStyle,
    SphereStyle,
    Visualizer,
)


## 1. Scenes

Create a `Visualizer`, add a `Point`, a `Line`, and a `Sphere`, and open the
interactive Three.js viewer. In a script the pattern is `show()` + `wait()`;
`display_snapshot()` gives a static, serverless inline preview. →
[Tutorial 02](../02_getting_started/)


In [ ]:
viz = Visualizer(title="Quick Tour — Scenes")

viz.add(Point(2, 0, 0), color="#ff4444", label="$P$")
viz.add(Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)), color="#44ff44")
viz.add(
    Sphere(Point(0, 0, 0), radius=1.5),
    style=SphereStyle(wireframe=True),
    opacity=0.3,
)

viz.display_snapshot()  # static, serverless inline preview


## 2. SDF Objects

Opt an entity into smooth, ray-marched signed-distance-field rendering with the
`SdfStyle` marker (WebGL2 required), and compose solids with `SdfObject` + the
Python CSG operators. → [Tutorial 03](../03_sdf_objects/)


In [ ]:
viz = Visualizer(title="Quick Tour — SDF", add_default_axes=False, add_default_grid=False)
viz.add(Sphere(Point(0, 0, 0), 1.0), color="#4477cc")                      # normal mesh
viz.add(Sphere(Point(2.5, 0, 0), 1.1), style=SdfStyle(color="#ffaa00"))     # ray-marched
viz.display_snapshot()


## 3. Scene Graphs & Transforms

Group entities into hierarchies with `VizGroup`, mutate them by reference with
`VizObjectRef`, and transform them. → [Tutorial 05](../05_scene_graphs/)


In [ ]:
viz = Visualizer(title="Quick Tour — Scene Graph", add_default_axes=False, add_default_grid=False)
grp = viz.add_group("spinner")
grp.new(Point(0, 0, 0), color="#ff4444", label="hub")
grp.new(Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)), color="#44aaff")
grp.set_transform(rotation=(0.0, 0.0, 0.7))
viz.flush()
viz.display_snapshot()


## 4. Styles & Colors

Give entities color, opacity, wireframe, and point size via the per-type style
dataclasses (`SphereStyle`, `PointStyle`, …). → [Tutorial 06](../06_styles_colors/)


In [ ]:
viz = Visualizer(title="Quick Tour — Styles", add_default_axes=False, add_default_grid=False)
viz.add(Point(0, 0, 0), color="#ffcc00", style=PointStyle(size=0.2), label="origin")
viz.add(
    Sphere(Point(3, 0, 0), 1.2),
    style=SphereStyle(wireframe=True, wireframe_color="#00ffff", opacity=0.3),
)
viz.display_snapshot()


## 5. Axes, Grid & Camera

Frame the scene with axes/grid and configure the camera. A default set is
inserted per scene unless disabled. Passing a `View2DConfig` camera deduces a 2D
viewer automatically. → [Tutorial 07](../07_axes_grid_camera/)


In [ ]:
from pytanga.viz import View2DConfig

viz = Visualizer(
    title="Quick Tour — 2D camera",
    camera=View2DConfig(xmin=0, xmax=8, ymin=0, ymax=6),
)
viz.add(Point(3, 4, 0), color="#ff4444", label="$P(3,4)$")
viz.display_snapshot()


## 6. Plotting

Build a complete 2D/3D plotting coordinate system (grid, axes, value labels,
plots) with `CoordinateSystem`. → [Tutorial 08](../08_coordinate_system/)


In [ ]:
import math

viz = Visualizer(
    title="Quick Tour — Plot",
    space_dim=2,
    add_default_axes=False,
    add_default_grid=False,
)
cs = CoordinateSystem(viz, xlim=(0, 2 * math.pi), ylim=(-1.5, 1.5), labels=("x", "sin(x)"))
xs = [0.05 * i for i in range(int(2 * math.pi / 0.05) + 1)]
cs.plot(xs, [math.sin(x) for x in xs], color="#44ff44", style=PointPathStyle(line_thickness=3))
viz.display_snapshot()


## 7. Labels & Annotations

Add labels, titles, and Markdown/LaTeX annotations (rendered via KaTeX). →
[Tutorial 09](../09_labels/)


In [ ]:
viz = Visualizer(title="Quick Tour — Labels", annotation="A sphere $S$ with a label.")
viz.add(Sphere(Point(0, 0, 0), 1.5), opacity=0.4, label="$S$")
viz.add(
    Point(0, 0, 0),
    color="#ffff00",
    label="Origin",
    label_style=LabelStyle(offset_local=(0, 1.1, 0)),
)
viz.display_snapshot()


## 8. Interaction

Make objects clickable and draggable with the pointer-event system
(`InteractionConfig` / `InteractionTrigger` / `InteractionEventType`). →
[Tutorial 10](../10_interaction/)


In [ ]:
from pytanga.viz import InteractionConfig, InteractionEventType, InteractionTrigger

viz = Visualizer(title="Quick Tour — Interaction", add_default_axes=False, add_default_grid=False)
pid = viz.add(Point(0, 0, 0), color="#ff4444", label="click me")
viz.set_interaction(
    pid,
    InteractionConfig(
        enabled=True,
        triggers=[InteractionTrigger(InteractionEventType.CLICK)],
    ),
)

async def on_click(event):
    print("clicked at", event.world_position)

viz.on_interaction(pid, InteractionEventType.CLICK, on_click)
viz.flush()
viz.display_snapshot()


## 9. Animation

Animate entities with `Visualizer.animate()` (call `show()` first), `PointPath`,
`animate_to`, and `Timeline`. → [Tutorial 11](../11_animation/)


In [ ]:
import math

viz = Visualizer(title="Quick Tour — Animation", add_default_axes=False, add_default_grid=False)
p = viz.new(Point(3, 0, 0), color="#ff4444", label="orbit")

angle = 0.0
for _ in viz.animate(fps=30):
    angle += 0.1
    p.entity = Point(3 * math.cos(angle), 3 * math.sin(angle), 0)
    viz.flush()
    if angle > 2 * math.pi:
        break

viz.stop_server()
viz.display_snapshot()


## 10. Split Views & Layouts

Show several scenes and control panels in one browser page with
`SplitView`/`SceneView`/`GroupView` and `show(layout=...)`. →
[Tutorial 12](../12_split_views/)


In [ ]:
from pytanga.viz import SceneView, SplitView

viz = Visualizer(add_default_axes=False, add_default_grid=False)
side = viz.scene("side")
side.add(Point(2, 0, 0), color="#44ff44")

layout = SplitView("horizontal", [SceneView(""), SceneView("side")])
# viz.show(layout=layout)   # opens both panes at a single URL
print("built layout:", layout)


## 11. Interactive Apps

Build an interactive `VisualizerApp` with sliders, dropdowns, and buttons, using
the managed lifecycle (`init` → block → `cleanup`). →
[Tutorial 13](../13_visualizer_app/)


In [ ]:
from pytanga.viz import ControlEvent, VisualizerApp

class TourApp(VisualizerApp):
    async def init(self):
        self.viz.add(Sphere(Point(0, 0, 0), 1.0), entity_id="s", color="#4488ff", opacity=0.4)
        self.viz.add_slider("r", label="Radius", min=0.2, max=3.0, value=1.0, on_change=self.on_r)
        self.viz.flush()

    async def on_r(self, value: float, _event: ControlEvent):
        self.viz.update_entity("s", Sphere(Point(0, 0, 0), value))
        self.viz.flush()

# TourApp().run()  # managed lifecycle: init → block → cleanup
print("TourApp defined — run it in tutorial 13.")


## 12. Controls

Use every control type (slider, dropdown, button, text field, color picker,
checkbox, …) and in-place value updates. → [Tutorial 14](../14_controls/)


In [ ]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)
viz.add_slider("s", label="Slider", min=0, max=1, value=0.5)
viz.add_dropdown("d", label="Dropdown", options=["a", "b"], value="a")
viz.add_button("b", label="Button")
viz.add_text_field("t", label="Text", value="hello")
viz.add_checkbox("c", label="Checkbox", value=True)
viz.add_color_picker("cp", label="Color", value="#4488ff")
viz.flush()
print("controls added to the scene")


## 13. Banners & Dialogs

Show status overlays, prompts, and modal dialogs with
`show_banner()` / `alert()` / `confirm()`. → [Tutorial 15](../15_banners_dialogs/)


In [ ]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)
viz.add(Point(0, 0, 0), color="#ff4444")

# In a running viewer these render as overlays:
viz.show_banner("## Welcome\n\nThis is a banner.", title="Notice")
viz.confirm("Proceed?")
viz.alert("Done.")
print("banners queued")


## 14. Responsive Computation

Keep the viewer responsive during long-running work with `flush_async()` and
compute offload (`submit_user`). → [Tutorial 16](../16_responsive_computation/)


In [ ]:
# Inside an async handler (VisualizerApp):
#   self.viz.set_annotation("Calculating…")
#   await self.viz.flush_async()          # render before blocking
#   ...
#   self.submit_user(_work, done=_done)   # run work off the event loop
print("flush_async() awaits a flush; submit_user() offloads work (tutorial 16).")


## 15. Export

Export self-contained HTML, embeddable figures, and glTF/GLB — all read from the
in-memory scene (no server needed). → [Tutorial 17](../17_export/)


In [ ]:
viz = Visualizer(title="Quick Tour — Export")
viz.add(Sphere(Point(0, 0, 0), 1.5), style=SphereStyle(wireframe=True), opacity=0.4)
viz.add(Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)), color="#44ff44")
viz.add(Point(2, 0, 0), color="#ff4444", label="P")

viz.export_snapshot("_output/01_quick_tour_figure.html", overwrite=True)
print("exported _output/01_quick_tour_figure.html")


## Visual Examples

A single standalone HTML figure generated by `pytanga.viz.Visualizer` — a styled
sphere, a line, and a label — exported via `export_snapshot()`.


In [ ]:
viz = Visualizer(title="Quick Tour — Figure")
viz.add(
    Sphere(Point(0, 0, 0), radius=2.0),
    style=SphereStyle(wireframe=True, wireframe_color="#66ccff"),
    opacity=0.35,
    label="$S_1$",
)
viz.add(Line(origin=Point(-3, 0, 0), direction=Direction(1, 0, 0)), color="#44ff44", label="x-axis")
viz.add(Point(2, 1, 0), color="#ff4444", style=PointStyle(size=0.15), label="$P$")

viz.display_snapshot()   # render inline (serverless)
viz.export_snapshot("_output/01_quick_tour_figure.html", overwrite=True)
print("standalone figure written to _output/01_quick_tour_figure.html")


## Summary & next steps

You have seen the whole viewer at a glance:

| Capability | Dive deeper |
|---|---|
| Scenes | [02 — Getting Started](../02_getting_started/) |
| SDF objects | [03 — SDF Objects](../03_sdf_objects/) |
| Scene graphs | [05 — Scene Graphs](../05_scene_graphs/) |
| Styles & colors | [06 — Styles & Colors](../06_styles_colors/) |
| Axes, grid & camera | [07 — Axes, Grid & Camera](../07_axes_grid_camera/) |
| Plotting | [08 — Coordinate System](../08_coordinate_system/) |
| Labels | [09 — Labels](../09_labels/) |
| Interaction | [10 — Interaction](../10_interaction/) |
| Animation | [11 — Animation](../11_animation/) |
| Split views | [12 — Split Views](../12_split_views/) |
| Interactive apps | [13 — VisualizerApp](../13_visualizer_app/) |
| Controls | [14 — Controls](../14_controls/) |
| Banners & dialogs | [15 — Banners & Dialogs](../15_banners_dialogs/) |
| Responsive computation | [16 — Responsive Computation](../16_responsive_computation/) |
| Export | [17 — Export](../17_export/) |
| GA entities | [18 — GA Entities](../18_ga_entities/) |
| SDF viewer | [19 — SDF Viewer](../19_sdf_viewer/) |
